In [1]:
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
from sklearn.svm import LinearSVC

RANDOM_SEED = 42

# Locate the project root.
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "diversevul_stratified_10000.csv"
)

print("SVM model development environment ready.")
print("Project root:", PROJECT_ROOT)
print("Dataset path:", DATA_PATH)
print("Dataset exists:", DATA_PATH.exists())

Matplotlib is building the font cache; this may take a moment.


SVM model development environment ready.
Project root: f:\BSc (Hons) in Cyber Security\Projects\NLP_Group_02
Dataset path: f:\BSc (Hons) in Cyber Security\Projects\NLP_Group_02\data\processed\diversevul_stratified_10000.csv
Dataset exists: True


In [2]:
# Load the processed working dataset.

svm_df = pd.read_csv(
    DATA_PATH,
    usecols=["func", "target"]
)

# Final validation before model development.
svm_df = svm_df.dropna(
    subset=["func", "target"]
).copy()

svm_df["func"] = svm_df["func"].astype(str)
svm_df["target"] = svm_df["target"].astype(int)

print("Dataset loaded successfully.")
print("Dataset shape:", svm_df.shape)

print("\nClass distribution:")
print(
    svm_df["target"]
    .value_counts()
    .sort_index()
)

print("\nMissing values:")
print(svm_df.isna().sum())

print("\nDuplicate source-code functions:")
print(svm_df["func"].duplicated().sum())

assert len(svm_df) == 10_000
assert set(svm_df["target"].unique()) == {0, 1}

print("\nDataset validation completed successfully.")

Dataset loaded successfully.
Dataset shape: (10000, 2)

Class distribution:
target
0    9409
1     591
Name: count, dtype: int64

Missing values:
func      0
target    0
dtype: int64

Duplicate source-code functions:
0

Dataset validation completed successfully.


In [3]:
# Separate the source code and target labels.

X = svm_df["func"]
y = svm_df["target"]

# First split:
# 70% training and 30% temporary data.
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=RANDOM_SEED,
    stratify=y,
)

# Second split:
# Divide temporary data equally into validation and testing sets.
X_validation, X_test, y_validation, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=RANDOM_SEED,
    stratify=y_temp,
)

print("Dataset split completed successfully.")

print("\nTraining set:")
print("Records:", len(X_train))
print(y_train.value_counts().sort_index())

print("\nValidation set:")
print("Records:", len(X_validation))
print(y_validation.value_counts().sort_index())

print("\nTesting set:")
print("Records:", len(X_test))
print(y_test.value_counts().sort_index())

print("\nTotal records after splitting:")
print(len(X_train) + len(X_validation) + len(X_test))

assert len(X_train) == 7000
assert len(X_validation) == 1500
assert len(X_test) == 1500
assert (
    len(X_train)
    + len(X_validation)
    + len(X_test)
    == len(svm_df)
)

print("\nSplit validation completed successfully.")

Dataset split completed successfully.

Training set:
Records: 7000
target
0    6586
1     414
Name: count, dtype: int64

Validation set:
Records: 1500
target
0    1411
1      89
Name: count, dtype: int64

Testing set:
Records: 1500
target
0    1412
1      88
Name: count, dtype: int64

Total records after splitting:
10000

Split validation completed successfully.


In [4]:
# Tokenise C/C++ source code into identifiers, numbers,
# operators and punctuation symbols.

import re

CODE_TOKEN_PATTERN = re.compile(
    r"""
    0x[0-9A-Fa-f]+
    |
    [A-Za-z_]\w*
    |
    \d+(?:\.\d+)?
    |
    ==|!=|<=|>=|->|\+\+|--|&&|\|\||<<|>>
    |
    [{}()\[\];,.+\-*/%&|^~!<>=?:]
    """,
    re.VERBOSE,
)


def code_tokenizer(source_code):
    """Convert one source-code function into code-related tokens."""
    return CODE_TOKEN_PATTERN.findall(str(source_code))


# Demonstrate the tokenizer using a short example.
example_code = """
int add_numbers(int a, int b) {
    return a + b;
}
"""

print("Example tokens:")
print(code_tokenizer(example_code))

Example tokens:
['int', 'add_numbers', '(', 'int', 'a', ',', 'int', 'b', ')', '{', 'return', 'a', '+', 'b', ';', '}']


In [5]:
# Convert source-code tokens into TF-IDF features.
# The vectoriser is fitted only on training data to prevent data leakage.

tfidf_vectorizer = TfidfVectorizer(
    tokenizer=code_tokenizer,
    token_pattern=None,
    lowercase=False,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.98,
    max_features=50_000,
    sublinear_tf=True,
    dtype=np.float32,
)

print("Fitting TF-IDF vectoriser on training data...")

X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)

print("Transforming validation data...")
X_validation_tfidf = tfidf_vectorizer.transform(X_validation)

print("Transforming testing data...")
X_test_tfidf = tfidf_vectorizer.transform(X_test)

print("\nTF-IDF feature extraction completed successfully.")

print("\nTraining feature shape:")
print(X_train_tfidf.shape)

print("\nValidation feature shape:")
print(X_validation_tfidf.shape)

print("\nTesting feature shape:")
print(X_test_tfidf.shape)

print("\nVocabulary size:")
print(len(tfidf_vectorizer.vocabulary_))

assert X_train_tfidf.shape[0] == 7000
assert X_validation_tfidf.shape[0] == 1500
assert X_test_tfidf.shape[0] == 1500

print("\nTF-IDF validation completed successfully.")

Fitting TF-IDF vectoriser on training data...
Transforming validation data...
Transforming testing data...

TF-IDF feature extraction completed successfully.

Training feature shape:
(7000, 50000)

Validation feature shape:
(1500, 50000)

Testing feature shape:
(1500, 50000)

Vocabulary size:
50000

TF-IDF validation completed successfully.
